In [23]:
import torch
import pandas as pd
import torchvision
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import os
from PIL import Image
from pathlib import Path
import config
import numpy as np
import copy

In [13]:
#数据预处理

#检查json结构
import json
train_path="./train/annotations_classification.json"

def inspect_json(path):
    with open(path,encoding="utf-8")as f:
        #加载数据
        data=json.load(f)

    print("顶层类型：", type(data).__name__) 

    if isinstance(data,dict):
        #key返回的只是一个对象，无法索引
        print("顶层key:{}".format(list(data.keys())))

        for k,v in data.items():
            if isinstance(v,list):
                print(f"{k},列表长度:{len(v)}")
                if v:
                    print(f"第一个元素类型: {type(v[0]).__name__}")
                    print(f"样例: {str(v[0])[:200]}")
            elif isinstance(v,dict):
                print(f"{k},{v}:dict,key:{list(v.keys())[:10]}")
    elif isinstance(data,list):
        print(f"列表长度:{len(data)}")
        if data:
            print(f"示例:{str(data[0])[:200]}")
            if isinstance(data[0],dict):
                print(f"{list(data[0].keys())[:10]}")

    return data

data=inspect_json(train_path)


顶层类型： dict
顶层key:['categories', 'images', 'image_labels']
categories,列表长度:10
第一个元素类型: dict
样例: {'id': 1, 'name': 'airplane'}
images,列表长度:585
第一个元素类型: dict
样例: {'id': 0, 'file_name': 'images/001.jpg', 'width': 958, 'height': 808}
image_labels,列表长度:585
第一个元素类型: dict
样例: {'image_id': 0, 'category_ids': [1]}


In [14]:
print(type(data))
'''
顶层是对象,且不等长数组,分开处理
'''

<class 'dict'>


'\n顶层是对象,且不等长数组,分开处理\n'

In [15]:
categories=pd.DataFrame(data["categories"])
train_images=pd.DataFrame(data["images"])
images_labels=pd.DataFrame(data["image_labels"])
#确定转换成功
print(type(categories),type(train_images))
#观察形状
print(categories.shape)
print(train_images.shape)
print(images_labels.shape)
#检测
print(categories.iloc[:,:])
print(train_images.iloc[:5,:])
print(images_labels.iloc[:5,:])

<class 'pandas.core.frame.DataFrame'> <class 'pandas.core.frame.DataFrame'>
(10, 2)
(585, 4)
(585, 2)
   id                name
0   1            airplane
1   2                ship
2   3        storage_tank
3   4    baseball_diamond
4   5        tennis_court
5   6    basketball_court
6   7  ground_track_field
7   8              harbor
8   9              bridge
9  10             vehicle
   id       file_name  width  height
0   0  images/001.jpg    958     808
1   1  images/002.jpg    950     806
2   2  images/003.jpg    889     803
3   3  images/004.jpg    946     732
4   4  images/005.jpg    966     753
   image_id category_ids
0         0          [1]
1         1          [1]
2         2          [1]
3         3          [1]
4         4          [1]


In [17]:
#数据处理
#删除id列
categories=categories.iloc[:,1:]    #这个应该只是用来最后的通过下标告诉这个到底是什么类别的
#合并,舍去，转为onehot
# train_images=pd.concat([train_images,images_labels],axis=0)

#对于标签转为独热编码
one_hot=[]
for label in images_labels["category_ids"].to_numpy():
    tmp=[0]*config.num_catogeries
    for idx in label:
        tmp[idx-1]=1
    one_hot.append(tmp)
train_images["one_hot"]=one_hot


In [18]:
#裁剪/缩放图片
min_width=min(train_images.to_numpy()[:,2])
min_height=min(train_images.to_numpy()[:,3])
print(min_width)    #533
print(min_height)   #358
#太小了，说明可能得检测整个分布，，不能一刀切破坏图片信息
max_width=max(train_images.to_numpy()[:,2])
max_height=max(train_images.to_numpy()[:,3])
print(max_width)    #1728
print(max_height)   #1028
#继续查看其他特征

# print(images.to_numpy().mean())
#检测缺失值
# num_na=0
# for x in images.to_numpy()[:,2]:
#     if not isinstance(x,int):
#         num_na=num_na+1
# print(num_na)   #无缺失值

print(train_images.to_numpy()[:,2].mean())    #937.8752136752137
print(train_images.to_numpy()[:,3].mean())    #609.2854700854701

num_900=0
for x in train_images.to_numpy()[:,2]:
    if x<600:
        num_900+=1
print(num_900)

533
358
1728
1028
937.8752136752137
609.2854700854701
4


In [ ]:
#方向不对，不应该直接裁剪
#宽高比例
train_images["ratio"]=train_images["width"]/train_images["height"]
print(train_images["ratio"].describe())
print(train_images["ratio"].value_counts(bins=10).sort_index())

#ratio集中在其中的几个区域，说明数据集中，容易缩放
#随机裁剪+统一比例
train_tf=torchvision.transforms.Compose([
    #随机裁剪 + 缩放比例
    torchvision.transforms.RandomResizedCrop(224,scale=(0.7,1.0)),
    torchvision.transforms.ToTensor(),
    #归一化
    torchvision.transforms.Normalize([0.485, 0.456, 0.406],
                                    [0.229, 0.224, 0.225])
])
#不能随机裁剪
test_tf=torchvision.transforms.Compose([
    torchvision.transforms.Resize(256),        # 短边缩到 256
    torchvision.transforms.CenterCrop(224),    # 中心裁 224
    torchvision.transforms.ToTensor(),
    #归一化
    torchvision.transforms.Normalize([0.485, 0.456, 0.406],
                                    [0.229, 0.224, 0.225])
])
#图片数据集
class MY_DATASET(Dataset):
    def __init__(self, df:pd.DataFrame , dir,tf):
        self.images_dir=dir
        self.df=df.reset_index(drop=True)   #不懂
        self.transform=tf

    def __len__(self):
        return len(self.df)

    def __getitem__(self,idx):
        #按整数取行
        row=self.df.iloc[idx]
        path=os.path.join(self.images_dir,row["filename"])
        img=Image.open(path).convert("RGB")
        #实时转换
        img=self.transform(img)
        #独热编码,这里要转为tensor
        label = torch.tensor(row["one_hot"], dtype=torch.float32)

        return img,label

class MY_DATASET_NO_LABEL(Dataset):
    def __init__(self,dir,tf):
        self.img_dir=dir
        self.transform=tf
        # 枚举所有图片-----重点
        self.files = sorted([
            p.name for p in Path(self.img_dir).glob("*")
            if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self,idx):
        name=self.files[idx]
        path=os.path.join(self.img_dir,self.files[idx])
        img=Image.open(path).convert("RGB")
        img=self.transform(img)

        return img,name

train_dataset=MY_DATASET(train_images,"./train/images",train_tf)
test_dataset=MY_DATASET_NO_LABEL("./test/images",test_tf)
print(test_dataset[0])
#dataloader
# def train_dataloader(train_dataset):
#     train_dataloader=DataLoader(
#         dataset=train_dataset,
#         batch_size=64,
#         shuffle=True,
#         num_workers=0,
#         drop_last=False
#     )
#     return train_dataloader



count    585.000000
mean       1.557593
std        0.235350
min        0.836066
25%        1.421687
50%        1.536028
75%        1.681750
max        2.823529
Name: ratio, dtype: float64
ratio
(0.833, 1.035]      5
(1.035, 1.234]     30
(1.234, 1.432]    118
(1.432, 1.631]    233
(1.631, 1.83]     146
(1.83, 2.029]      31
(2.029, 2.227]     15
(2.227, 2.426]      5
(2.426, 2.625]      1
(2.625, 2.824]      1
Name: count, dtype: int64
(tensor([[[-0.7137, -0.5938, -0.5082,  ...,  0.5022,  0.7762,  0.7933],
         [-0.6794, -0.6109, -0.6623,  ...,  0.8447,  0.8104,  0.7077],
         [-0.5596, -0.6794, -0.8335,  ...,  0.8618,  0.7419,  0.2453],
         ...,
         [-1.0219, -1.0048, -0.9705,  ..., -0.8507, -0.9192, -0.8849],
         [-1.0219, -1.0219, -0.9877,  ..., -0.9192, -0.9363, -0.8678],
         [-1.0219, -1.0219, -1.0048,  ..., -0.9534, -0.9363, -0.7650]],

        [[-0.5651, -0.4426, -0.3550,  ...,  0.8704,  1.1155,  1.0630],
         [-0.5301, -0.4601, -0.5126,  ...,  1.

完成了数据的处理和打包

In [ ]:
from torchvision import models
from torch import nn
import time

#迁移学习，使用预训练网络
# 加载预训练模型,使用第一版权重，更稳定
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# 改最后一层，输出改成 10 类
num_features = model.fc.in_features   # ResNet18 是 512
model.fc = nn.Linear(num_features, 10)

# 放到 GPU
if torch.cuda.is_available():
    model = model.to("cuda")

#多指标分类使用 BCE ，内置了 sigmoid,默认是mean
loss_fn=nn.BCEWithLogitsLoss()
if torch.cuda.is_available():
    loss_fn=loss_fn.cuda()

#评价指标
from torchmetrics.classification import MultilabelAveragePrecision

def macro_MAP(num_labels,preds,target):
    """
    preds是最终的logits,target是指标
    """
    metric = MultilabelAveragePrecision(num_labels=num_labels, average='macro')

    # 计算 Macro-mAP
    map_score = metric(preds, target)
    return map_score

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\mouseco/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


In [ ]:
def train(train_df,train_dir,train_tf,valid_df,valid_dir,valid_tf,
          model:nn.Module,loss,num_epochs,learning_rate,weight_decay):
    """返回每轮的loss和验证集上的metric指标已经训练好的网络"""
    #深拷贝网络
    net = copy.deepcopy(model)

    #评估时候对全部数据进行求loss
    train_ls,valid_ls=[],[]

    updater=torch.optim.Adam(
        net.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )
    train_dataset=MY_DATASET(train_df,train_dir,train_tf)
    valid_dataset=MY_DATASET(valid_df,valid_dir,valid_tf)

    train_dataloader=DataLoader(train_dataset,batch_size=64,shuffle=True,num_workers=0,drop_last=False)
    valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False, num_workers=0)

    for epoch in range(num_epochs):
        net.train()
        #小批量训练
        for data in train_dataloader:
            imgs,labels=data
            logits=net(imgs)
            ls=loss(logits,labels)
            updater.zero_grad()
            ls.mean().backward()
            updater.step()

        #评估
        net.eval()
        with torch.no_grad():
            #完整训练集每轮对于整个训练集的loss
            train_ls_sum=0
            for data in train_dataloader:
                imgs,labels=data
                train_logits=net(imgs)
                train_ls_sum+=loss(train_logits,labels).item()*imgs.shape[0]
            train_ls.append(train_ls_sum)

            #验证集的loss曲线和macro-MAP指标
            valid_ls_sum=0
            for data in valid_loader:
                imgs,labels=data
                valid_logits=net(imgs)
                valid_ls_sum+=loss(valid_logits,labels).item()*imgs.shape[0]
            valid_ls.append(valid_ls_sum)
            #评估指标
            metric=macro_MAP(config.num_catogeries,valid_logits,labels)

    return train_ls,valid_ls,net,metric


In [ ]:
#k折实验
# from sklearn.model_selection import StratifiedKFold
#多指标分类器
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from d2l import torch as d2l

#拆分
# skf = StratifiedKFold(config.n_splits, shuffle=True, random_state=42)
mskf = MultilabelStratifiedKFold(config.n_splits, shuffle=True, random_state=42)

train_l_sum,valid_l_sum=0,0
fold_model=[]
fold_score=[]
#拆分成分布类似的集合
for fold,(train_idx, valid_idx) in enumerate(mskf.split(train_images, np.array(train_images["one_hot"].tolist()))):
    train_df=train_images.iloc[train_idx].reset_index(drop=True)
    valid_df=train_images.iloc[valid_idx].reset_index(drop=True)

    #每一折前都要重新取模型
    

    train_dir="./train/images"
    train_ls , valid_ls , net_trained , metric = train(train_df,train_dir,train_tf,
                                                       valid_df,train_dir,test_tf,
                                                       model,loss_fn,config.num_epochs,
                                                       config.learning_rate,config.weight_decay)
    #截取每个模型的最后一轮的loss
    train_l_sum+=train_ls[-1]
    valid_l_sum+=valid_ls[-1]
    #每一折后都作图
    d2l.plot(list(range(1, config.num_epochs + 1)), [train_ls, valid_ls],
                xlabel='epoch', ylabel='BCE', xlim=[1, config.num_epochs],
                legend=['train', 'valid'], yscale='log')
    print(f'折{fold + 1},训练loss:{float(train_ls[-1]):f}, '
              f'验证loss:{float(valid_ls[-1]):f}')
    #模型评估记录
    fold_model.append(net_trained)
    fold_score.append(metric)

print(f'{config.n_splits}-折验证: 平均训练loss: {float(train_l_sum / config.n_splits):f}, '
      f'平均验证loss: {float(valid_l_sum / config.n_splits):f}')
for fold in range(config.n_splits):
    print(f"第{fold + 1}折模型的macro-MAP评估分数: {fold_score[fold]}")

In [ ]:
#不进行k折实验

def train_and_preds(train_df,train_dir,train_tf,valid_df,valid_dir,valid_tf,
          model:nn.Module,loss,num_epochs,learning_rate,weight_decay):
    """测试函数,对训练集和测试集全部测试,且训练集用macro-MAP进行评估模型"""
    #训练集直接训练
    train_ls , valid_ls , net_trained , metric = train(train_df,train_dir,train_tf,
                                                       valid_df,valid_dir,valid_tf,
                                                       model,loss,num_epochs,
                                                       learning_rate,weight_decay)
    print(f"macro-MAP评估分数: {metric}")
    #测试集测试,并组装 json
    net_trained.eval()
    with torch.no_grad():     #这些细节还是得加强理解
      test_dataloader=DataLoader(test_dataset,config.batch_size,shuffle=False,num_workers=0)  #这里有几个参数不懂
      for imgs in test_dataloader:
        test_output=net_trained(test_dataset).detach().numpy()
      #组装json
      ans=[]
      for i in range(len(test_dataset)):
         _,image_name=test_dataset[i]
         image_id=int(image_name.split('.')[0])
         score=test_output[i].tolist() #得到一个一维10个
         item = {
          "image_id": image_id,
          "score": score
          }
         ans.append(item)
         #其参数是中文正常显示，有合理的缩进，将python对象转为json字符串
      # json_str = json.dumps(ans, ensure_ascii=False, indent=2)
      with open("classification_predictions.json", "w", encoding="utf-8") as f:
        json.dump(ans, f, ensure_ascii=False, indent=2)

